# Experiment with 1080 method variants

The aim of these experiments is to find the models and parameters that perform best on AnthroSet (Lonke et al., 2025). Tested variables include:
- different masked language models
- different embedding models
- different values for *k* most probable tokens from an MLM to fill a mask
- different ways of extracting embeddings (static/contextual/template)
- short vs long context lenght
- different scoring methods


## Anthropomorphization Detection Pipeline

Supports multiple MLM and embedding models with correct
model-specific behavior (pooling, mask tokens, prefixes).

Flow:
  1. Load MLM via HF fill-mask pipeline
  2. Load embedding model with correct pooling per model card
  3. Build prototype vectors once at initialization
  4. For each sentence:

  - Get top-k candidates from MLM pipeline
  - Fill mask with all candidates, encode in one batched forward pass
  - Extract positional hidden state for each candidate
  - Score against prototype vectors via cosine similarity
  - Compute weighted human-likeness score and log-odds score (following AnthroScore)

In [ ]:
# ── 0. Imports ─────────────────────────────────────────────────────────────────
import torch
import torch.nn.functional as F
import numpy as np
from dataclasses import dataclass, field
from transformers import pipeline, Pipeline, AutoTokenizer, AutoModel, PreTrainedModel, PreTrainedTokenizer
from typing import Iterable, overload, Literal
from collections import OrderedDict
from itertools import product
import json
import pandas as pd
import time
import csv

In [ ]:
# ── 1. Constants ────────────────────────────────────────────────────────────────

_prototype_cache = {}

ANIMATE_DESCRIPTIONS = [
    "a living person, human being, or individual who can act and think",
    "a human who makes decisions and has intentions",
    "someone who thinks, feels, and can take action",
    "a name of a person or a personal pronoun referring to a human",
    "a person with a professional role such as a doctor, scientist, or engineer",
    "a human subject or participant in a study or experiment",
    "an individual who experiences emotions and has subjective awareness",
    "a named historical or contemporary figure such as an author or inventor",
    "a collective group of humans such as a team, committee, or community",
    "a pronoun referring to one or more people such as I, you, he, she, or we",
    "a human agent who performs actions and bears responsibility",
    "a living creature capable of perception and voluntary movement",
    "an animate being with a social role or identity",
    "a person referred to by their job title, role, or name"
]

INANIMATE_DESCRIPTIONS = [
    "a non-living machine, device, tool, or system",
    "a piece of software or automated system without consciousness",
    "an object or artifact that cannot think or feel",
    "a technical system or piece of equipment",
    "an abstract concept or a thing that does not have a physical form",
    "a physical object with no capacity for thought or intention",
    "an automated process or algorithm that executes without awareness",
    "a product, platform, or application built by humans",
    "a natural or man-made structure such as a building or infrastructure",
    "a measurement, metric, or quantitative value",
    "a document, report, or piece of written output",
    "an institution or organization treated as a non-human entity",
    "a chemical substance, material, or physical phenomenon",
    "a vehicle, vessel, or means of transport",
    "something that is acted upon rather than acting of its own will",
    "a pronoun referring to an inanimate object such as it"
]

CLASS_DESCRIPTIONS = [ANIMATE_DESCRIPTIONS, INANIMATE_DESCRIPTIONS]
CLASS_NAMES_DEFAULT = ["animate", "inanimate"]

# Cache the static embeddings in format {(model_id, prefix, template, token): np.ndarray}
_static_emb_cache: dict[tuple[str, str, str, str], np.ndarray] = {}

MAX_CACHED_MODELS = 5
# Cache the models to avoid redownloads
_model_cache = OrderedDict()

MLM_IDS = (
    "distilbert/distilbert-base-uncased",
    "google-bert/bert-base-uncased",
    "google-bert/bert-large-uncased",
    "FacebookAI/roberta-base",
    "answerdotai/ModernBERT-base"
)

EMB_IDS = (
    "sentence-transformers/all-MiniLM-L6-v2",
    "BAAI/bge-base-en-v1.5",
    "nomic-ai/modernbert-embed-base",
    "jinaai/jina-embeddings-v5-text-nano"
)

MODEL_PAIRS = list(product(MLM_IDS, EMB_IDS))


# ── 2. Model registry ───────────────────────────────────────────────────────────
# Pooling strategy and any required input prefix, sourced from official model cards.

EMB_MODEL_CONFIGS = {
    # mean pool — no prefix
    "sentence-transformers/all-MiniLM-L6-v2":  {"pooling": "mean", "query_prefix": "", "doc_prefix": "", "static_template": "What is the definition of [PLACEHOLDER]?"},
    # mean pool - prefix
    "nomic-ai/modernbert-embed-base":          {"pooling": "mean", "query_prefix": "search_query: ", "doc_prefix": "search_document: ", "static_template": "What is the most accurate description of [PLACEHOLDER]?"},
    # CLS pool — prefix with a single space (<-- NOTE! Changed from an earlier version)
    "BAAI/bge-base-en-v1.5":                   {"pooling": "cls",  "query_prefix": " ", "doc_prefix": "", "static_template": "Can you define [PLACEHOLDER]?"},
    # last token pool - prefix
    "jinaai/jina-embeddings-v5-text-nano":     {"pooling": "last", "query_prefix": "Query: ", "doc_prefix": "Document: ", "static_template": "Can you define [PLACEHOLDER]?"},
}

def get_emb_config(model_id: str) -> dict:
    if model_id in EMB_MODEL_CONFIGS:
        return EMB_MODEL_CONFIGS[model_id]
    # Fallback: warn and use mean pooling with no prefix
    print(f"  ⚠ '{model_id}' not in registry — defaulting to mean pooling, no prefix, no template.")
    print(f"    Check the model card and add it to EMB_MODEL_CONFIGS if needed.")
    return {"pooling": "mean", "query_prefix": "", "doc_prefix": "", "static_template": ""}

In [ ]:
# ── 3. Config dataclass ──────────────────────────────────────────────────────

@dataclass
class Config:
    # MLM
    mlm_pipe:             Pipeline  # HF fill-mask pipeline — to automatically handle model-specific settings
    mlm_model_id:         str

    # Embedding model
    emb_tokenizer:        PreTrainedTokenizer
    emb_model:            PreTrainedModel
    emb_model_id:         str
    emb_config:           dict  # {"pooling": "mean"|"cls", "query_prefix": str, "doc_prefix": str}

    # Prototype setup
    class_descriptions:   list[list[str]]
    class_names:          list[str]
    class_protos:         np.ndarray = field(init=False, default_factory=lambda: np.empty(0)) # (n_classes, hidden_dim)

    mask_token:           str = field(init=False, default="") # [MASK] for BERTs, <mask> for RoBERTa
    template_placeholder: str = "[PLACEHOLDER]"

    def build_prototype_vectors(self):
        """
        Embed each class description list, average per class, re-normalize.
        Returns (class_protos, class_names).
        """

        if not self.emb_model_id in _prototype_cache:

            print("Building prototype vectors…")
            protos = []
            for descriptions in self.class_descriptions:
                embs  = embed_texts_batch(descriptions, self, "doc_prefix") # (n_desc, hidden_dim)
                proto = embs.mean(axis=0)
                proto = proto / (np.linalg.norm(proto) + 1e-12) # re-normalize back to unit length
                protos.append(proto)

            _prototype_cache[self.emb_model_id] = np.array(protos)
            print("Prototype vectors cached.")

        return _prototype_cache[self.emb_model_id] # (n_classes, hidden_dim)

    def __post_init__(self):
        self.class_protos = self.build_prototype_vectors()
        print("Prototype vectors loaded.")

        self.mask_token = self.mlm_pipe.tokenizer.mask_token
        print(f"Model uses {self.mask_token} as mask_token.")


# ── 4. Pooling functions ───────────────────────────────────────────────────────

def normalize_and_move(pooled: torch.Tensor) -> np.ndarray:
    normalized = F.normalize(pooled)
    return normalized.detach().cpu().numpy()

def pool_batch(hidden: torch.Tensor,
               attention_mask: torch.Tensor,
               strategy: str) -> np.ndarray:
    """
    Pool a batch of hidden states to one vector per sequence.
    hidden: (batch, seq_len, hidden_dim)
    Returns: (batch, hidden_dim) numpy array, L2-normalized per row.
    """
    if strategy == "cls":
        pooled = hidden[:, 0, :]                            # (batch, seq_len, hidden_dim)
    elif strategy == "mean":
        mask   = attention_mask.unsqueeze(-1).float()       # (batch, seq_len, 1)
        summed = (hidden * mask).sum(dim=1)                 # (batch, hidden_dim)
        counts = mask.sum(dim=1).clamp(min=1e-12)           # (batch, 1)
        pooled = summed / counts                            # (batch, hidden_dim)
    elif strategy == "last":
        sequence_lengths = attention_mask.sum(dim=1) - 1      # (batch,)
        batch_idx        = torch.arange(hidden.size(0), device=hidden.device)
        pooled           = hidden[batch_idx, sequence_lengths] # (batch, hidden_dim)
    else:
        raise ValueError(f"Unknown pooling strategy: {strategy!r}")

    return normalize_and_move(pooled)

def embed_texts_batch(texts: list[str], config: Config, prefix_key: str) -> np.ndarray:
    """
    Embed a list of texts in one batched forward pass.
    Applies model-specific prefix before encoding.
    Returns: (n_texts, hidden_dim) numpy array.
    """
    prefix        = config.emb_config[prefix_key]
    strategy      = config.emb_config["pooling"]
    prefixed      = [prefix + t for t in texts] if len(prefix) > 0 else texts

    device        = next(config.emb_model.parameters()).device
    inputs        = config.emb_tokenizer(
                        prefixed,
                        return_tensors="pt",
                        truncation=True,
                        max_length=config.emb_model.config.max_position_embeddings,
                        padding=True
                    )
    inputs        = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        hidden    = config.emb_model(**inputs).last_hidden_state

    return pool_batch(hidden, inputs["attention_mask"], strategy)

def embed_texts_batch_cached(tokens: list[str], config: Config, prefix_key: str, template: str = "") -> np.ndarray:
    """
    Embed tokens statically, using cache for tokens seen before.
    Returns: (n_tokens, hidden_dim)
    """
    cache_key            = config.emb_model_id
    prefix               = config.emb_config[prefix_key]
    template_placeholder = config.template_placeholder
    uncached = [t for t in tokens if (cache_key, prefix, template, t) not in _static_emb_cache]

    if uncached:
        if len(template) > 0:
            embs = embed_texts_batch([template.replace(template_placeholder, t) for t in uncached], config, prefix_key) # one batch for new tokens
        else:
            embs = embed_texts_batch(uncached, config, prefix_key)
        for token, emb in zip(uncached, embs):
            _static_emb_cache[(cache_key, prefix, template, token)] = emb

    return np.array([_static_emb_cache[(cache_key, prefix, template, t)] for t in tokens])


# ── 5. Initialization ──────────────────────────────────────────────────────────

def load_pipeline(mlm_model_id:  str,
                  emb_model_id:  str,
                  class_descriptions: list[list[str]] = CLASS_DESCRIPTIONS,
                  class_names:   list[str] = CLASS_NAMES_DEFAULT,
                  device: str = "cuda") -> Config:
    """
    Load both models once, build prototype vectors, return a Config object
    that stores all the necessary information.
    """

    @overload
    def get_or_load(model_id: str, model_type: Literal["mlm"]) -> Pipeline: ...

    @overload
    def get_or_load(model_id: str, model_type: Literal["emb"]) -> tuple[PreTrainedTokenizer, PreTrainedModel]: ...

    def get_or_load(model_id: str, model_type: str) -> Pipeline | tuple[PreTrainedTokenizer, PreTrainedModel]:
        if model_id in _model_cache:
            _model_cache.move_to_end(model_id)  # mark as recently used
            return _model_cache[model_id]

        if len(_model_cache) >= MAX_CACHED_MODELS:
            evicted_id, _ = _model_cache.popitem(last=False)  # remove oldest
            print(f"Evicting {evicted_id} from cache")

        if model_type == "mlm":
          print(f"Loading MLM: {mlm_model_id}")
          _model_cache[model_id] = pipeline(
                                    "fill-mask",
                                    model=model_id,
                                    device=device
                                  )
          return _model_cache[model_id]

        elif model_type == "emb":
            print(f"Loading embedding model: {emb_model_id}")
            emb_tokenizer = AutoTokenizer.from_pretrained(emb_model_id,
                                                          trust_remote_code=True)
            emb_model     = AutoModel.from_pretrained(emb_model_id,
                                                      trust_remote_code=True,
                                                      attn_implementation="sdpa")
            emb_model.eval()
            if device == "cuda" and torch.cuda.is_available():
                emb_model = emb_model.cuda()
            _model_cache[emb_model_id] = (emb_tokenizer, emb_model)
            return (emb_tokenizer, emb_model)

        else: raise ValueError(model_type)


    mlm_pipe                 = get_or_load(mlm_model_id, "mlm")
    emb_config               = get_emb_config(emb_model_id)
    emb_tokenizer, emb_model = get_or_load(emb_model_id, "emb")

    config = Config(
        mlm_pipe=mlm_pipe,
        mlm_model_id=mlm_model_id,
        emb_tokenizer=emb_tokenizer,
        emb_model=emb_model,
        emb_model_id=emb_model_id,
        emb_config=emb_config,
        class_descriptions=class_descriptions,
        class_names=class_names
    )
    print("Ready.\n")
    return config


# ── 6. MLM prediction ──────────────────────────────────────────────────────────

def predict_mask(sentences: Iterable[str],
                 config: Config,
                 top_k: int = 50,
                 batch_size = 128):
    """
    Return top-k candidate tokens and their MLM probabilities.
    Uses HF fill-mask pipeline in order to handle mask token and
    tokenization correctly for any supported MLM (BERT, RoBERTa, ModernBERT, etc.).
    """

    if isinstance(sentences, str):
        sentences = [sentences]
    results = config.mlm_pipe(sentences, top_k=top_k, batch_size=batch_size)

    for result in results:
      tokens    = [r["token_str"].strip() for r in result]
      mlm_probs = np.array([r["score"] for r in result])
      yield {"tokens": tokens, "mlm_probs": mlm_probs}


# ── 7. Batched contextual embedding ───────────────────────────────────────────

def embed_candidates_in_context(masked_sentence: str,
                                result: dict,
                                config: Config,
                                verbose: bool = False) -> np.ndarray:
    """
    Gets a contextual embedding for each candidate token by finding the
    token position in the filled sentence and extracting the embedding for
    that(/those) position(s).
    Assumes the text contains a single [MASK] token.
    """

    mask_start = masked_sentence.index(config.mask_token)
    mask_end   = mask_start + len(config.mask_token)
    pre, post  = masked_sentence[:mask_start], masked_sentence[mask_end:]

    if verbose:
      print(f"{repr(masked_sentence)} (start={mask_start}, end={mask_end}, span={masked_sentence[mask_start:mask_end]})")

    filled = []
    token_spans = []
    for token in result["tokens"]:
      sent = pre + token + post
      filled.append(sent)
      token_end = mask_start + len(token)
      token_spans.append((mask_start, token_end))

      if verbose:
        span = sent[mask_start:token_end]
        print(f"token='{token}', sent='{sent}' (start={mask_start}, end={token_end}, span='{span}')")
        if span != token:
          print(f"WARNING: {token} ≠ {span}")

    # Add a prefix and embed the filled sentences with an embedding model
    prefix     = config.emb_config["query_prefix"]
    prefix_len = len(prefix)
    prefixed   = [prefix + r for r in filled] if prefix_len > 0 else filled

    assert all((a == b[prefix_len:] for a, b in zip(filled, prefixed))), "Problem aligning the strings"

    token_spans_shifted = np.array(token_spans, dtype=np.int64) + prefix_len

    device = next(config.emb_model.parameters()).device
    batch  = config.emb_tokenizer(
                prefixed,
                return_tensors="pt",
                truncation=True,
                max_length=config.emb_model.config.max_position_embeddings,
                padding=True,
                return_offsets_mapping=True)

    offset_mapping = batch.pop("offset_mapping")  # (n_candidates, seq_len, 2), stays on CPU
    input_ids = batch["input_ids"].clone()
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        hidden = config.emb_model(**batch).last_hidden_state  # (n_candidates, seq_len, hidden_dim)

    # Build a per-candidate token mask: which token positions overlap the candidate's char span
    span_mask = torch.zeros(offset_mapping.shape[:2], dtype=torch.bool)  # (n_candidates, seq_len)
    for i, (start, end) in enumerate(token_spans_shifted):
        starts = offset_mapping[i, :, 0]
        ends   = offset_mapping[i, :, 1]
        # special tokens ([CLS]/[SEP]/pad) have offset (0, 0) -> excluded automatically
        overlap = (starts < end) & (ends > start) & (ends > starts)
        span_mask[i] = overlap

    if verbose:
        print("\n[NEW IMPLEMENTATION]\nContextual embedding extracted for tokens:")
        grouped_tokens = [
            config.emb_tokenizer.convert_ids_to_tokens(input_ids[i][span_mask[i]])
            for i in range(input_ids.shape[0])
        ]
        print(grouped_tokens)

    span_mask = span_mask.unsqueeze(-1).to(hidden.device).float()   # (n_candidates, seq_len, 1)
    token_counts = span_mask.sum(dim=1).clamp(min=1)                # handle span not found

    if verbose:
      if (token_counts > 1).any():
        print()
        multi_span_indices = torch.where(token_counts > 1)[0]
        print("multi-span tokens", [result["tokens"][i] for i in multi_span_indices])
        print()

    pooled = (hidden * span_mask).sum(dim=1) / token_counts
    return normalize_and_move(pooled)

def embed_candidates_in_context_old(sentence: str,
                                    tokens: list[str],
                                    config: Config,
                                    verbose: bool = False) -> np.ndarray:
    """
    Fill mask with each candidate token, run all filled sentences through
    the embedding model in a batched forward pass, and extract the
    hidden state at the candidate token's position.

    Position is computed once from the first candidate (all candidates
    occupy the same position since MLM outputs are always single tokens
    and WordPiece tokenization is position-independent).

    (Problem: can snap to incorrect position, and does not work with BPE
    tokenization.)

    Returns: (n_candidates, hidden_dim) numpy array, L2-normalized.
    """
    mask_token     = config.mask_token
    prefix         = config.emb_config["query_prefix"]
    strategy       = config.emb_config["pooling"]

    # Build all filled sentences with embedding model prefix
    filled = [prefix + sentence.replace(mask_token, t) for t in tokens]

    # Find target position once using the first candidate
    probe_inputs  = config.emb_tokenizer(filled[0], return_tensors="pt")
    token_strings = config.emb_tokenizer.convert_ids_to_tokens(
                        probe_inputs["input_ids"][0])
    first_subword = config.emb_tokenizer.tokenize(tokens[0])
    first_subword = first_subword[0] if first_subword else None

    target_pos = None
    if first_subword:
        for i, t in enumerate(token_strings[1:], start=1):
            if t == first_subword:
                target_pos = i
                break

    # Batch tokenize all filled sentences
    device      = next(config.emb_model.parameters()).device
    batch       = config.emb_tokenizer(
                      filled, return_tensors="pt",
                      truncation=True, max_length=config.emb_model.config.max_position_embeddings, padding=True)
    batch       = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        hidden  = config.emb_model(**batch).last_hidden_state
    # hidden: (n_candidates, max_seq_len, hidden_dim)

    if target_pos is None or target_pos >= hidden.shape[1]:
        # Fallback: pool the full sentence
        embs = pool_batch(hidden, batch["attention_mask"], strategy)
        if verbose:
            print("\n[OLD IMPLEMENTATION]\nUsed pooling instead of contextual embedding")
    else:
        if verbose:
            print(f"\n[OLD IMPLEMENTATION]\nfirst_subword={first_subword}, target_pos={target_pos}")
            print("Contextual embedding extracted for tokens:")
            extracted_tokens = []
            for i in range(batch["input_ids"].shape[0]):
                token_id = batch["input_ids"][i][target_pos].item()   # 0-d tensor -> python int
                extracted_tokens.append(config.emb_tokenizer.convert_ids_to_tokens(token_id))
            print(extracted_tokens)
        # Extract target position for all candidates simultaneously
        embs  = hidden[:, target_pos, :].detach().cpu().numpy()
        norms = np.linalg.norm(embs, axis=1, keepdims=True) + 1e-9
        embs  = embs / norms

    return embs   # (n_candidates, hidden_dim)

# ── 8. Scoring ─────────────────────────────────────────────────────────────────


def score_candidates(embeddings: np.ndarray,
                     config: Config,
                     verbose: bool = False) -> np.ndarray:
    """
    Compute animacy class similarities for each candidate embedding.
    Get the difference between the similarities by subtracting the similarity
    to the inanimate prototype from the similarity to the animate prototype.
    Positive values indicate the embedding is closer to the animate prototype,
    and negative values indicate the embedding is closer to the inanimate prototype.
    embeddings: (n_candidates, hidden_dim)
    """
    # (n_candidates, hidden_dim) @ (hidden_dim, n_classes) → (n_candidates, n_classes)
    sims  = embeddings @ config.class_protos.T
    sim_diff = sims[:, 0] - sims[:, 1]
    if verbose:
        print(sim_diff)
        print()
    return sim_diff

def scores_from(sim_diff: np.ndarray,
                mlm_probs: np.ndarray,
                top_k: int) -> dict:
    sim_diff   = sim_diff[:top_k]
    probs      = mlm_probs[:top_k]
    total_prob = probs.sum()

    soft_score     = float(np.dot(sim_diff, probs) / total_prob) \
                     if total_prob > 0 else 0.0

    animate_prob   = probs[sim_diff > 0].sum() + 1e-12
    inanimate_prob = probs[sim_diff <= 0].sum() + 1e-12
    hard_score     = float(np.log(animate_prob / inanimate_prob))

    return {
        "soft": soft_score,
        "hard": hard_score
    }

# ── 9. Full classify_mask ──────────────────────────────────────────────────────

def classify_mask(sentences: Iterable[str],
                  config: Config,
                  metadata: dict,
                  top_k_values: list[int] = [1, 5, 10, 50, 100, 150],
                  batch_size: int = 128,
                  embedding_types: list[str] = ["static"],
                  verbose: bool = False) -> list:

    allowed_embedding_types = set(("static", "template", "contextual", "contextual_old"))
    for emb_type in embedding_types:
      if emb_type not in allowed_embedding_types:
        raise ValueError(f"Unknown embedding type: {emb_type!r}\nAllowed values are: {list(sorted(allowed_embedding_types))}")

    mlm = metadata.get("mlm")
    emb = metadata.get("emb")
    long_cont = metadata.get("long_cont")

    max_k = max(top_k_values)
    outputs = []

    for i, (sentence, result) in enumerate(zip(
          sentences,
          predict_mask(sentences, config, top_k=max_k, batch_size=batch_size)
      )):

        if i % 100 == 0:
            print(f"Processing document at index {i}")

        if verbose:
          print("-"*20)

        sim_diff = {}
        for emb_type in embedding_types:
          if emb_type == "static":
            if verbose: print("static")
            embeddings = embed_texts_batch_cached(result["tokens"], config, "query_prefix")
          elif emb_type == "template":
            embeddings = embed_texts_batch_cached(result["tokens"], config, "query_prefix", config.emb_config["static_template"])
            if verbose: print("template")
          elif emb_type == "contextual":
            embeddings = embed_candidates_in_context(sentence, result, config, verbose=verbose)
            if verbose: print("\ncontextual")
          elif emb_type == "contextual_old":
            embeddings = embed_candidates_in_context_old(sentence, result["tokens"], config, verbose=verbose)
            if verbose: print("contextual old")

          sim_diff[emb_type] = score_candidates(embeddings, config, verbose=verbose)

        # Collect scores from different top-k:s
        all_scores = {"i": i}
        for top_k in top_k_values:
          for emb_type, diff in sim_diff.items():
            scores = scores_from(diff, result["mlm_probs"], top_k)
            for metric, score in scores.items():
                all_scores[f"{metric}_{mlm}_{emb}_k{top_k}_{emb_type}_{long_cont}"] = score

        outputs.append(all_scores)

        if verbose:
            for k, v in all_scores.items():
              print(f"{k}: {v}")
            print()

    return outputs

In [ ]:
# ── 10. Example usage ──────────────────────────────────────────────────────────

def get_short_name(model_id: str) -> str:
  return model_id.rsplit("/", 1)[-1]

EXAMPLES = [
    "[MASK] reached optimum proficiency in tasks after just one repetition.",
    "[MASK] invented the telephone.",
    "[MASK] was installed on the server last night.",
    "[MASK] decided to escalate the issue to management.",
    "[MASK] crashed unexpectedly during the experiment.",
    "[MASK] wrote a letter to the editor.",
    "After the update, [MASK] didn't work as it used to.",
    "She felt seen while talking to [MASK].",
    "A deceptive [MASK] tried to fool the human.",
    "We think the output of [MASK] is hallucinated.",
    "We think [MASK] hallucinates.",
    "[MASK] told the doctor about the hallucinations they had.",
    "Hallucination is a know quirk in [MASK].",
    "I am using [MASK] as a tutor.",
    "I am using an [MASK] tutor."
  ]

def run_examples(
  examples: list[str] = EXAMPLES,
  mlm_model_id = MLM_IDS[0],
  emb_model_id = EMB_IDS[0],
  device="cpu",
  top_k_values=[10],
  embedding_types=["static", "template", "contextual", "contextual_old"],
  return_outputs: bool = False
):
    test_config = load_pipeline(
        mlm_model_id = mlm_model_id,
        emb_model_id = emb_model_id,
        device=device
    )

    if "roberta" in test_config.mlm_model_id:
        examples = [ex.replace("[MASK]", "<mask>") for ex in examples]

    outputs = classify_mask(examples, test_config, verbose=True, top_k_values=top_k_values,
                            embedding_types=embedding_types,
                            metadata={"mlm": get_short_name(test_config.mlm_model_id), "emb": get_short_name(test_config.emb_model_id), "long_cont": 0})

    if return_outputs:
      return outputs

## Run examples

In [ ]:
run_examples(device="cuda", top_k_values=[1, 20], emb_model_id="BAAI/bge-base-en-v1.5")

Loading MLM: distilbert/distilbert-base-uncased


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Building prototype vectors…
Prototype vectors cached.
Prototype vectors loaded.
Model uses [MASK] as mask_token.
Ready.

Processing document at index 0
--------------------
static
[ 0.07031786  0.04261124  0.09220278  0.08840489  0.0444814   0.0129441
  0.00970185  0.03179193  0.06652689  0.08966398  0.04795295 -0.04515374
 -0.04022503 -0.0237754  -0.01614988  0.03802031  0.04174638  0.06136012
  0.05385536  0.00588608]

template
[ 0.08595365  0.06461513  0.07526422  0.08739376  0.04802895 -0.00302559
  0.03413892  0.06779522  0.07191354  0.0936799   0.05405408 -0.02657187
 -0.04276848  0.01175165 -0.03673708  0.06089097  0.06076682  0.0766995
  0.04445934 -0.00113261]

'[MASK] reached optimum proficiency in tasks after just one repetition.' (start=0, end=6, span=[MASK])
token='participants', sent='participants reached optimum proficiency in tasks after just one repetition.' (start=0, end=12, span='participants')
token='students', sent='students reached optimum proficiency in tasks aft

## Upload AnthroSet

In [ ]:
df = pd.read_csv("anthroset_extended.csv")
sample = df.groupby("ling_category").sample(1, random_state=12345)

In [ ]:
test_sentences = sample["masked_sentence_bert"].tolist()

## Test BAAI/bge-base-en-v1.5 with the added prefix

In [ ]:
run_examples(test_sentences, emb_model_id="BAAI/bge-base-en-v1.5", device="cuda", top_k_values=[1, 5, 10], embedding_types=["static", "contextual"])

Prototype vectors loaded.
Model uses [MASK] as mask_token.
Ready.

Processing document at index 0
--------------------
static
[ 0.10532153  0.02951598 -0.0580864  -0.06271887 -0.03560233  0.09715432
  0.09054321  0.02279669  0.00424379  0.07829058]

'The results demonstrate the descriptive value of the ResQu model to predict behavior and perceptions of responsibility by considering the characteristics of the human, the intelligent [MASK], the environment and some systematic behavioral biases.' (start=184, end=190, span=[MASK])
token='person', sent='The results demonstrate the descriptive value of the ResQu model to predict behavior and perceptions of responsibility by considering the characteristics of the human, the intelligent person, the environment and some systematic behavioral biases.' (start=184, end=190, span='person')
token='##sia', sent='The results demonstrate the descriptive value of the ResQu model to predict behavior and perceptions of responsibility by considering the ch

## Run  BAAI/bge-base-en-v1.5

In [ ]:
results_filename = "results_bge_new.jsonl"
runtime_filename = "runtimes_bge_new.jsonl"

emb_model_id = "BAAI/bge-base-en-v1.5"

def run_anthroset_bge(
  df: pd.DataFrame,
  device="cuda",
  top_k_values=[1, 5, 10, 50, 100, 150],
  embedding_types=["static"],
  verbose: bool = False
):

  with open(results_filename, "w") as file, open(runtime_filename, "w") as runtime_file:
    for mlm_model_id in MLM_IDS:

        print("-"*30)
        print(f"{mlm_model_id} + {emb_model_id}")

        config = load_pipeline(
            mlm_model_id = mlm_model_id,
            emb_model_id = emb_model_id,
            device=device
        )

        for long_cont in (0, 1):
          if long_cont == 0:
            col_name = "masked_sentence"
          else:
            col_name = "long_context"

          if config.mask_token == "<mask>":
            pass

          elif config.mask_token == "[MASK]":
            col_name += "_bert"

          else:
            raise ValueError(f"No text field available for {config.mask_token} as mask token")

          texts = df[col_name].tolist()
          metadata = {"mlm": get_short_name(mlm_model_id), "emb": get_short_name(emb_model_id), "long_cont": long_cont}

          start = time.time()
          anthroset_results = classify_mask(texts, config, verbose=verbose,
                                            top_k_values=top_k_values, embedding_types=embedding_types,
                                            metadata=metadata)
          end = time.time() - start
          metadata["runtime"] = end
          print(f"{mlm_model_id} + {emb_model_id} + {col_name} took {end // 60:.0f}m {end % 60:.0f}s\n")

          runtime_file.write(json.dumps(metadata))
          runtime_file.write("\n")

          for res in anthroset_results:
            file.write(json.dumps(res))
            file.write("\n")

In [ ]:
run_anthroset_bge(df)

------------------------------
distilbert/distilbert-base-uncased + BAAI/bge-base-en-v1.5
Prototype vectors loaded.
Model uses [MASK] as mask_token.
Ready.

Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400
Processing document at index 500
distilbert/distilbert-base-uncased + BAAI/bge-base-en-v1.5 + masked_sentence_bert took 0m 21s

Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400
Processing document at index 500
distilbert/distilbert-base-uncased + BAAI/bge-base-en-v1.5 + long_context_bert took 0m 31s

------------------------------
google-bert/bert-base-uncased + BAAI/bge-base-en-v1.5
Loading MLM: google-bert/bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

... (24 outputs left out)

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Prototype vectors loaded.
Model uses [MASK] as mask_token.
Ready.



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400
Processing document at index 500
answerdotai/ModernBERT-base + BAAI/bge-base-en-v1.5 + masked_sentence_bert took 0m 18s

Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400
Processing document at index 500
answerdotai/ModernBERT-base + BAAI/bge-base-en-v1.5 + long_context_bert took 0m 36s



## Test all models (excluding jina)

In [ ]:
def test_all_models():
  for mlm_id, emb_id in MODEL_PAIRS:
    print("-"*30)
    print(f"Using models {mlm_id} + {emb_id}")
    run_examples(test_sentences, mlm_id, emb_id, "cuda", top_k_values=[1, 5, 10])

In [ ]:
test_all_models()

------------------------------
Using models distilbert/distilbert-base-uncased + sentence-transformers/all-MiniLM-L6-v2
Prototype vectors loaded.
Model uses [MASK] as mask_token.
Ready.

Processing document at index 0
--------------------
static
[ 0.29534256  0.0675867  -0.15570766 -0.06753784 -0.1233753   0.28701583
  0.16983989 -0.01455772  0.06413335  0.18698925]

template
[ 0.2551166  -0.04196118 -0.22890955 -0.11616138 -0.18158388  0.19199342
  0.14287072 -0.06726795 -0.00088388  0.08937541]

'The results demonstrate the descriptive value of the ResQu model to predict behavior and perceptions of responsibility by considering the characteristics of the human, the intelligent [MASK], the environment and some systematic behavioral biases.' (start=184, end=190, span=[MASK])
token='person', sent='The results demonstrate the descriptive value of the ResQu model to predict behavior and perceptions of responsibility by considering the characteristics of the human, the intelligent person, 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Prototype vectors loaded.
Model uses [MASK] as mask_token.
Ready.

Processing document at index 0
--------------------
static
[ 0.10532153  0.02951598 -0.0580864  -0.06271887 -0.03560233  0.09715432
  0.09054321  0.02279669  0.00424379  0.07829058]

template
[ 0.11452633 -0.02530074 -0.0783335  -0.05927539 -0.05091351  0.08893573
  0.08482993  0.00614148 -0.01973766  0.0702697 ]

'The results demonstrate the descriptive value of the ResQu model to predict behavior and perceptions of responsibility by considering the characteristics of the human, the intelligent [MASK], the environment and some systematic behavioral biases.' (start=184, end=190, span=[MASK])
token='person', sent='The results demonstrate the descriptive value of the ResQu model to predict behavior and perceptions of responsibility by considering the characteristics of the human, the intelligent person, the environment and some systematic behavioral biases.' (start=184, end=190, span='person')
token='##sia', sent='The res

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Prototype vectors loaded.
Model uses [MASK] as mask_token.
Ready.

Processing document at index 0
--------------------
static
[ 0.2309615  -0.01997396 -0.09123853 -0.05542219 -0.03261627  0.19462249
  0.17504498  0.00765084  0.01815672  0.13189715]

template
[ 0.13835683 -0.03964958 -0.08051914 -0.04566455 -0.04767311  0.13866043
  0.10579807  0.00268313  0.00631526  0.10538799]

'The results demonstrate the descriptive value of the ResQu model to predict behavior and perceptions of responsibility by considering the characteristics of the human, the intelligent [MASK], the environment and some systematic behavioral biases.' (start=184, end=190, span=[MASK])
token='person', sent='The results demonstrate the descriptive value of the ResQu model to predict behavior and perceptions of responsibility by considering the characteristics of the human, the intelligent person, the environment and some systematic behavioral biases.' (start=184, end=190, span='person')
token='##sia', sent='The res

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

... (5 outputs left out)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Prototype vectors loaded.
Model uses <mask> as mask_token.
Ready.

Processing document at index 0
--------------------
static
[-0.03629291 -0.15570766  0.28701583  0.29534256 -0.00072277  0.05232447
  0.0310932  -0.06753784  0.03350291  0.12549835]

template
[-0.07417256 -0.22890955  0.19199342  0.2551166  -0.05801228 -0.00253788
 -0.02646047 -0.11616138 -0.03428164  0.07968354]

'The results demonstrate the descriptive value of the ResQu model to predict behavior and perceptions of responsibility by considering the characteristics of the human, the intelligent <mask>, the environment and some systematic behavioral biases.' (start=184, end=190, span=<mask>)
token='brain', sent='The results demonstrate the descriptive value of the ResQu model to predict behavior and perceptions of responsibility by considering the characteristics of the human, the intelligent brain, the environment and some systematic behavioral biases.' (start=184, end=189, span='brain')
token='system', sent='The resul

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

Prototype vectors loaded.
Model uses [MASK] as mask_token.
Ready.

Processing document at index 0
--------------------
static
[ 0.21802098 -0.15570766 -0.11420801 -0.06753784 -0.10868561  0.13247636
  0.03350291 -0.14535552 -0.21073931 -0.06617641]

template
[ 0.1502161  -0.22890955 -0.2152746  -0.11616138 -0.18804392  0.09296647
 -0.03428161 -0.21892715 -0.23918241 -0.07703066]

'The results demonstrate the descriptive value of the ResQu model to predict behavior and perceptions of responsibility by considering the characteristics of the human, the intelligent [MASK], the environment and some systematic behavioral biases.' (start=184, end=190, span=[MASK])
token='agent', sent='The results demonstrate the descriptive value of the ResQu model to predict behavior and perceptions of responsibility by considering the characteristics of the human, the intelligent agent, the environment and some systematic behavioral biases.' (start=184, end=189, span='agent')
token='system', sent='The resul

## Evaluation functions

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

f1_cats = {'verb_subjects', 'verb_objects', 'adjectives'}
accuracy_cats = {'role_function_NPs', 'genitive_NPs'}

In [ ]:
def score_to_prediction(score: float) -> int:
  if score > 0:
    return 1
  return 0

def evaluate(output):
  preds_per_method = {k: [] for k in output[0] if k != "i"}
  for result_dict in output:
    for key, score in result_dict.items():
      if key == "i":
        continue
      preds_per_method[key].append(score_to_prediction(score))
  return preds_per_method

In [ ]:
def run_anthroset(
  examples: list[str] = df.masked_sentence_bert.tolist(),
  mlm_model_id = MLM_IDS[0],
  emb_model_id = EMB_IDS[0],
  device="cpu",
  top_k_values=[1, 5, 10, 50, 100, 150],
  verbose: bool = False
):
    test_config = load_pipeline(
        mlm_model_id = mlm_model_id,
        emb_model_id = emb_model_id,
        device=device
    )

    if "roberta" in test_config.mlm_model_id:
        examples = [ex.replace("[MASK]", "<mask>") for ex in examples]

    return classify_mask(examples, test_config, verbose=verbose, top_k_values=top_k_values,
                  metadata={"mlm": get_short_name(test_config.mlm_model_id), "emb": get_short_name(test_config.emb_model_id), "long_cont": 0})

full_output = run_anthroset(device="cuda")

Prototype vectors loaded.
Model uses [MASK] as mask_token.
Ready.

Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400
Processing document at index 500


## Run all models

(Except for jina which has problems with downloading)

In [ ]:
results_filename = "results_contextual_new.jsonl"
runtime_filename = "runtimes_contextual_new.jsonl"

def run_anthroset_all_models(
  df: pd.DataFrame,
  device="cuda",
  top_k_values=[1, 5, 10, 50, 100, 150],
  embedding_types=["contextual"],
  verbose: bool = False
):

  with open(results_filename, "w") as file, open(runtime_filename, "w") as runtime_file:
    for mlm_model_id, emb_model_id in MODEL_PAIRS:

        print("-"*30)
        print(f"{mlm_model_id} + {emb_model_id}")

        config = load_pipeline(
            mlm_model_id = mlm_model_id,
            emb_model_id = emb_model_id,
            device=device
        )

        for long_cont in (0, 1):
          if long_cont == 0:
            col_name = "masked_sentence"
          else:
            col_name = "long_context"

          if config.mask_token == "<mask>":
            pass

          elif config.mask_token == "[MASK]":
            col_name += "_bert"

          else:
            raise ValueError(f"No text field available for {config.mask_token} as mask token")

          texts = df[col_name].tolist()
          metadata = {"mlm": get_short_name(mlm_model_id), "emb": get_short_name(emb_model_id), "long_cont": long_cont}

          start = time.time()
          anthroset_results = classify_mask(texts, config, verbose=verbose,
                                            top_k_values=top_k_values, embedding_types=embedding_types,
                                            metadata=metadata)
          end = time.time() - start
          metadata["runtime"] = end
          print(f"{mlm_model_id} + {emb_model_id} + {col_name} took {end // 60:.0f}m {end % 60:.0f}s\n")

          runtime_file.write(json.dumps(metadata))
          runtime_file.write("\n")

          for res in anthroset_results:
            file.write(json.dumps(res))
            file.write("\n")

In [ ]:
run_anthroset_all_models(df)

------------------------------
distilbert/distilbert-base-uncased + sentence-transformers/all-MiniLM-L6-v2
Evicting FacebookAI/roberta-base from cache
Loading MLM: distilbert/distilbert-base-uncased


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Prototype vectors loaded.
Model uses [MASK] as mask_token.
Ready.

Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400
Processing document at index 500
distilbert/distilbert-base-uncased + sentence-transformers/all-MiniLM-L6-v2 + masked_sentence_bert took 0m 51s

Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400
Processing document at index 500
distilbert/distilbert-base-uncased + sentence-transformers/all-MiniLM-L6-v2 + long_context_bert took 2m 33s

------------------------------
distilbert/distilbert-base-uncased + BAAI/bge-base-en-v1.5
Prototype vectors loaded.
Model uses [MASK] as mask_token.
Ready.

Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: google-bert/bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Prototype vectors loaded.
Model uses [MASK] as mask_token.
Ready.

Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400
Processing document at index 500
google-bert/bert-base-uncased + sentence-transformers/all-MiniLM-L6-v2 + masked_sentence_bert took 0m 54s

Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400
Processing document at index 500
google-bert/bert-base-uncased + sentence-transformers/all-MiniLM-L6-v2 + long_context_bert took 2m 29s

------------------------------
google-bert/bert-base-uncased + BAAI/bge-base-en-v1.5
Prototype vectors loaded.
Model uses [MASK] as mask_token.
Ready.

Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400
Processing docu

... (3 outputs left out)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Prototype vectors loaded.
Model uses <mask> as mask_token.
Ready.

Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400
Processing document at index 500
FacebookAI/roberta-base + sentence-transformers/all-MiniLM-L6-v2 + masked_sentence took 0m 52s

Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400
Processing document at index 500
FacebookAI/roberta-base + sentence-transformers/all-MiniLM-L6-v2 + long_context took 2m 21s

------------------------------
FacebookAI/roberta-base + BAAI/bge-base-en-v1.5
Prototype vectors loaded.
Model uses <mask> as mask_token.
Ready.

Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400
Processing document at index 500
FacebookAI

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

Prototype vectors loaded.
Model uses [MASK] as mask_token.
Ready.

Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400
Processing document at index 500
answerdotai/ModernBERT-base + sentence-transformers/all-MiniLM-L6-v2 + masked_sentence_bert took 0m 56s

Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400
Processing document at index 500
answerdotai/ModernBERT-base + sentence-transformers/all-MiniLM-L6-v2 + long_context_bert took 2m 26s

------------------------------
answerdotai/ModernBERT-base + BAAI/bge-base-en-v1.5
Prototype vectors loaded.
Model uses [MASK] as mask_token.
Ready.

Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400
Processing document a

In [ ]:
from collections import defaultdict

with open(results_filename) as file:
  jsonl_results = [json.loads(line) for line in file]

def parse_key(key: str):
  scoring_method, mlm, emb, topk, cont_emb, long_cont = key.split("_")
  return {"scoring_method": scoring_method,
          "mlm": mlm,
          "emb": emb,
          "topk": int(topk[1:]),
          "cont_emb": cont_emb,
          "long_cont": int(long_cont)}

res_dict = defaultdict(list)
for line in jsonl_results:
  i = line.pop("i")
  for key, value in line.items():
    new_dict = parse_key(key)
    new_dict["score"] = value
    new_dict["index"] = i
    res_dict[i].append(new_dict)

len(res_dict)

600

In [ ]:
pd.concat(
    {k: pd.DataFrame(v) for k, v in res_dict.items()}
).droplevel(1).to_csv("results_contextual_new.csv", index=False)

## A quick comparison of new vs old contextual embedding results

In [ ]:
res = pd.merge(df.copy(), pd.DataFrame(evaluate(full_output)), left_index=True, right_index=True)
res

,id,previous_sentence,current_sentence,masked_sentence,next_sentence,AI_phrase,masked_phrase,AI_entity,component,expectations,...,long_context_bert,ling_category,soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_0,hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_0,soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_old_0,hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_old_0,soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_static_0,hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_static_0,soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_template_0,hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_template_0
0,3_acl_7_46407_0,"In conclusion, our findings can achieve less s...",Language models (LMs) are vulnerable to exploi...,<mask> are vulnerable to exploitation for adve...,Training LMs for safety alignment is extensive...,Language models (LMs),Language models (LMs),model,vulnerable,2,...,"In conclusion, our findings can achieve less s...",adjectives,1,1,1,1,1,1,0,0
1,3_arx_2005.06620_1286415_1,Artificial intelligence (AI) enabled products ...,While governments and businesses are eager to ...,While governments and businesses are eager to ...,This article introduces one of the first inter...,these autonomous and intelligent systems,systems,system,"autonomous,intelligent",2,...,Artificial intelligence (AI) enabled products ...,adjectives,0,0,0,0,0,0,0,0
2,3_arx_2408.08878_2129059_6,We discovered prompting to be a useful but und...,Integrating LLMs into KE tasks needs to be min...,Integrating LLMs into KE tasks needs to be min...,"Given the limited ethical training, most knowl...",responsible AI,AI,AI,responsible,2,...,We discovered prompting to be a useful but und...,adjectives,1,1,1,1,0,1,0,0
3,3_arx_2402.05827_2003466_5,We focus on three key research questions.,RQ1: Can edited LLMs behave consistently resem...,RQ1: Can edited LLMs behave consistently resem...,RQ2: To what extent does the rephrasing of pro...,communicative AI,AI,AI,communicative,2,...,We focus on three key research questions. RQ1:...,adjectives,1,1,1,1,0,0,0,0
4,3_arx_1904.13086_1118126_10,Simply putting a human into the loop does not ...,The results demonstrate the descriptive value ...,The results demonstrate the descriptive value ...,The ResQu model is a new quantitative method t...,the intelligent system,system,system,intelligent,2,...,Simply putting a human into the loop does not ...,adjectives,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,1_arx_2010.09890_1366220_2,"In WAH, an AI agent needs to help a human-like...","To succeed, the AI agent needs to i) understan...","To succeed, the <mask> needs to i) understand ...","For this challenge, we build VirtualHome-Socia...",the AI agent,AI agent,agent,"understand,coordinate",1,...,"In WAH, an AI agent needs to help a human-like...",verb_subjects,1,1,1,1,1,1,1,1
596,1_acl_1477_40886_4,"Then, we conduct experiments on four popular t...",The experimental results showcase that ChatGPT...,The experimental results showcase that <mask> ...,We also found that ChatGPT hardly understands ...,ChatGPT,ChatGPT,ChatGPT,"demonstrate proficiency,struggle",1,...,"Then, we conduct experiments on four popular t...",verb_subjects,1,1,1,1,1,1,0,0
597,1_arx_2303.09387_1809184_9,Our overall assessment is that while some prog...,In the absence of a consensus definition and r...,In the absence of a consensus definition and r...,We argue that such manipulation poses a signif...,AI systems,AI systems,system,learn to manipulate,1,...,Our overall assessment is that while some prog...,verb_subjects,0,0,0,0,0,0,0,0
598,1_arx_2212.01681_1757858_4,I argue that LMs are models of intentional com...,When performing next word prediction given a t...,When performing next word prediction given a t...,These representations can in turn influence su...,an LM,LM,LM,infer,1,...,I argue that LMs are models of intentional com...,verb_subjects,1,1,1

In [ ]:
def score_cats(group, method_col):
    if group.name in accuracy_cats:
        return accuracy_score(group['expectations'], group[method_col])
    return f1_score(group['expectations'], group[method_col], average='macro')

scored = pd.DataFrame({
    method: res[res["expectations"] != 2].groupby('ling_category').apply(lambda g: score_cats(g, method), include_groups=False)
    for method in res.columns[-8:]
})
scored.index.name = 'category'
scored.style.background_gradient(axis=None)

,soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_0,hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_0,soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_old_0,hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_old_0,soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_static_0,hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_static_0,soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_template_0,hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_template_0
category,,,,,,,,
adjectives,0.701093,0.689850,0.701093,0.689850,0.734091,0.732766,0.667317,0.691854
genitive_NPs,0.733333,0.733333,0.733333,0.733333,0.783333,0.716667,0.616667,0.566667
role_function_NPs,0.885714,0.871429,0.857143,0.857143,0.700000,0.628571,0.271429,0.300000
verb_objects,0.738992,0.698469,0.738992,0.698469,0.755952,0.763978,0.786914,0.763665
verb_subjects,0.801329,0.785321,0.801329,0.785321,0.828947,0.786075,0.817092,0.817906


In [ ]:
scored.mean().sort_values(ascending=False)

,0
soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_0,0.772092
soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_old_0,0.766378
soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_static_0,0.760465
hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_0,0.755680
hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_old_0,0.752823
hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_static_0,0.725611
soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_template_0,0.631884
hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_template_0,0.628018


In [ ]:
res.to_csv("distilbert_all-mini_results.csv", index=False)

In [ ]:
modernbert_output = run_anthroset(mlm_model_id='distilbert/distilbert-base-uncased', emb_model_id='nomic-ai/modernbert-embed-base', device="cuda")

Prototype vectors loaded.
Model uses [MASK] as mask_token.
Ready.

Processing document at index 0
Processing document at index 100
Processing document at index 200
Processing document at index 300
Processing document at index 400
Processing document at index 500


In [ ]:
res = pd.merge(res.copy(), pd.DataFrame(evaluate(modernbert_output)), left_index=True, right_index=True)

In [ ]:
scored = pd.DataFrame({
    method: res[res["expectations"] != 2].groupby('ling_category').apply(lambda g: score_cats(g, method), include_groups=False)
    for method in res.columns[-16:]
})
scored.index.name = 'category'
scored.style.background_gradient(axis=None)

,soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_0,hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_0,soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_old_0,hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_old_0,soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_static_0,hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_static_0,soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_template_0,hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_template_0,soft_distilbert-base-uncased_modernbert-embed-base_k50_contextual_0,hard_distilbert-base-uncased_modernbert-embed-base_k50_contextual_0,soft_distilbert-base-uncased_modernbert-embed-base_k50_contextual_old_0,hard_distilbert-base-uncased_modernbert-embed-base_k50_contextual_old_0,soft_distilbert-base-uncased_modernbert-embed-base_k50_static_0,hard_distilbert-base-uncased_modernbert-embed-base_k50_static_0,soft_distilbert-base-uncased_modernbert-embed-base_k50_template_0,hard_distilbert-base-uncased_modernbert-embed-base_k50_template_0
category,,,,,,,,,,,,,,,,
adjectives,0.701093,0.689850,0.701093,0.689850,0.734091,0.732766,0.667317,0.691854,0.706356,0.697056,0.558036,0.558036,0.729394,0.727331,0.703846,0.679681
genitive_NPs,0.733333,0.733333,0.733333,0.733333,0.783333,0.716667,0.616667,0.566667,0.716667,0.616667,0.416667,0.416667,0.733333,0.650000,0.700000,0.550000
role_function_NPs,0.885714,0.871429,0.857143,0.857143,0.700000,0.628571,0.271429,0.300000,0.742857,0.728571,0.771429,0.757143,0.514286,0.500000,0.257143,0.242857
verb_objects,0.738992,0.698469,0.738992,0.698469,0.755952,0.763978,0.786914,0.763665,0.756081,0.772222,0.599870,0.597009,0.763978,0.772343,0.739820,0.755694
verb_subjects,0.801329,0.785321,0.801329,0.785321,0.828947,0.786075,0.817092,0.817906,0.800563,0.801329,0.550919,0.556282,0.819670,0.828446,0.809615,0.835248


In [ ]:
scored.style.background_gradient(cmap="coolwarm_r", axis=None)

,soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_0,hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_0,soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_old_0,hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_old_0,soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_static_0,hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_static_0,soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_template_0,hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_template_0,soft_distilbert-base-uncased_modernbert-embed-base_k50_contextual_0,hard_distilbert-base-uncased_modernbert-embed-base_k50_contextual_0,soft_distilbert-base-uncased_modernbert-embed-base_k50_contextual_old_0,hard_distilbert-base-uncased_modernbert-embed-base_k50_contextual_old_0,soft_distilbert-base-uncased_modernbert-embed-base_k50_static_0,hard_distilbert-base-uncased_modernbert-embed-base_k50_static_0,soft_distilbert-base-uncased_modernbert-embed-base_k50_template_0,hard_distilbert-base-uncased_modernbert-embed-base_k50_template_0
category,,,,,,,,,,,,,,,,
adjectives,0.701093,0.689850,0.701093,0.689850,0.734091,0.732766,0.667317,0.691854,0.706356,0.697056,0.558036,0.558036,0.729394,0.727331,0.703846,0.679681
genitive_NPs,0.733333,0.733333,0.733333,0.733333,0.783333,0.716667,0.616667,0.566667,0.716667,0.616667,0.416667,0.416667,0.733333,0.650000,0.700000,0.550000
role_function_NPs,0.885714,0.871429,0.857143,0.857143,0.700000,0.628571,0.271429,0.300000,0.742857,0.728571,0.771429,0.757143,0.514286,0.500000,0.257143,0.242857
verb_objects,0.738992,0.698469,0.738992,0.698469,0.755952,0.763978,0.786914,0.763665,0.756081,0.772222,0.599870,0.597009,0.763978,0.772343,0.739820,0.755694
verb_subjects,0.801329,0.785321,0.801329,0.785321,0.828947,0.786075,0.817092,0.817906,0.800563,0.801329,0.550919,0.556282,0.819670,0.828446,0.809615,0.835248


In [ ]:
scored.mean().sort_values(ascending=False)

,0
soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_0,0.772092
soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_old_0,0.766378
soft_distilbert-base-uncased_all-MiniLM-L6-v2_k50_static_0,0.760465
hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_0,0.755680
hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_contextual_old_0,0.752823
soft_distilbert-base-uncased_modernbert-embed-base_k50_contextual_0,0.744505
hard_distilbert-base-uncased_all-MiniLM-L6-v2_k50_static_0,0.725611
hard_distilbert-base-uncased_modernbert-embed-base_k50_contextual_0,0.723169
soft_distilbert-base-uncased_modernbert-embed-base_k50_static_0,0.712132
hard_distilbert-base-uncased_modernbert-embed-base_k50_static_0,0.695624


In [ ]:
res.to_csv("distilbert_all-mini_modernbert_results.csv", index=False)